# Google News Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook searches Google News and exports structured results for qualitative research. Enter search terms, select a time period and country, and download news headlines, sources, dates, and descriptions as CSV or Excel.

The notebook uses the `gnews` library to query Google News RSS feeds — no API key required. Results include article titles, publication dates, source publishers, descriptions, and URLs.

## Key Features

1. **No API Key Required**: Queries Google News directly via public RSS feeds
2. **Two Search Modes**: Quick search (up to 100 results) or extended date-range search (hundreds+ via monthly batching)
3. **Time Period Filtering**: Restrict results by preset period or custom date range
4. **Country & Language Selection**: Target news from specific countries and languages
5. **Top News Access**: Retrieve current top headlines without a search query
6. **Checkpoint & Combine**: Extended searches save batch files, then merge and deduplicate into one export
7. **Structured Export**: Download results as CSV or Excel for qualitative analysis

## Workflow

1. **Configure**: Enter search terms, select time period and country
2. **Fetch**: Retrieve news results from Google News
3. **Review**: Preview headlines and metadata in a table
4. **Export**: Download structured data for further analysis

## How Results Work

Google News RSS feeds return a **maximum of 100 results per query**. The notebook offers two modes:

- **Quick Search**: A single query returning up to 100 results. Best for recent news or exploratory searches.
- **Extended Search**: Breaks a custom date range into monthly windows, fetches up to 100 results per window, saves each batch as a checkpoint, then combines and deduplicates into one export. A 12-month range can yield up to 1,200 results (minus duplicates).

For research requiring even larger corpora, consider supplementing with dedicated news databases (LexisNexis, ProQuest, GDELT).

## Applications

This notebook supports research that involves analyzing media coverage — tracking how topics are framed in news, collecting public discourse for qualitative coding, identifying patterns in media attention, and contextualizing fieldwork within broader news narratives. The structured exports are formatted for use with qualitative analysis software and other tools in the AI Anthropology Toolkit.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using programmatic data retrieval to support media analysis in qualitative research. The notebook collects and structures news data but does not interpret it. Analytical judgment remains with the researcher.

**Important**: News articles are copyrighted material. This notebook retrieves metadata (titles, descriptions, dates, sources) and links. Respect publisher terms of service when using this data in research.

## Target Audience

Designed for anthropologists and qualitative researchers who need to collect and analyze news media — from graduate students conducting discourse analysis to research teams tracking media coverage of specific topics.

## Technical Approach

The notebook uses the **gnews** library to query Google News RSS feeds. It supports keyword search, time-based filtering, country/language selection, and top headline retrieval. All processing runs locally in the notebook.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install required Python packages and import libraries for Google News retrieval, data processing, and interactive configuration. Run this cell first to ensure all dependencies are available.

In [ ]:
# Install required packages
!pip install gnews pandas openpyxl ipywidgets -q

import os
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')

from gnews import GNews
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def results_to_rows(results):
    """Convert gnews results list to list of row dicts."""
    rows = []
    for r in results:
        pub = r.get('publisher', {})
        rows.append({
            'title': r.get('title', ''),
            'source': pub.get('title', '') if isinstance(pub, dict) else str(pub),
            'published': r.get('published date', ''),
            'description': r.get('description', ''),
            'url': r.get('url', ''),
        })
    return rows


def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    thin_border = Border(
        left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
        top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF'),
    )
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = thin_border
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
                cell.border = thin_border
    wb.save(filepath)


print("\u2713 All packages installed and libraries loaded successfully")
print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f4f0 Ready to configure your Google News search!")

## Search Configuration

Configure your Google News search using the interactive controls below. Set your search terms, time period, country, and export preferences.

In [ ]:
# Interactive Search Interface

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

instructions_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
instructions_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f4f0 Google News Explorer</h2>'
instructions_html += '<p><strong>Welcome!</strong> This notebook searches Google News and exports structured results for qualitative research.</p>'
instructions_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
instructions_html += '<ol>'
instructions_html += '<li><strong>Choose a mode:</strong> Quick Search (up to 100 results) or Extended Search (date range with monthly batching)</li>'
instructions_html += '<li><strong>Configure:</strong> Enter keywords, set filters, and select export format</li>'
instructions_html += '<li><strong>Fetch:</strong> Click the button to retrieve news results</li>'
instructions_html += '<li><strong>Export:</strong> Results download as a single file</li>'
instructions_html += '</ol>'
instructions_html += '<p style="margin-top: 10px; color: #8B8C89; font-size: 0.9em;">'
instructions_html += '<strong>Quick Search</strong> returns up to 100 results from a preset time period. '
instructions_html += '<strong>Extended Search</strong> breaks a date range into monthly windows (up to 100 per window), '
instructions_html += 'saves each batch, then combines and deduplicates into one export.</p>'
instructions_html += '</div>'

instructions = widgets.HTML(value=instructions_html)

# ── Search mode ──
mode_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Search Mode</h3>')

mode_toggle = widgets.ToggleButtons(
    options=[('Quick Search', 'quick'), ('Extended Search (date range)', 'extended')],
    value='quick',
    style={'button_width': '220px'}
)

# ── Keywords ──
search_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f50d Search</h3>')

query_input = widgets.Text(
    value='',
    placeholder='Enter search terms (leave empty for Top News in Quick mode)',
    description='Keywords:',
    layout=layout,
    style=style
)

query_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'Enter keywords to search Google News. In Quick mode, leave empty for top headlines.</p>'
)

# ── Quick mode filters ──
config_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c5 Filters</h3>')

period_input = widgets.Dropdown(
    options=[
        ('Past hour', '1h'),
        ('Past day', '1d'),
        ('Past week', '7d'),
        ('Past month', '30d'),
        ('Past year', '1y'),
    ],
    value='7d',
    description='Time period:',
    layout=layout,
    style=style
)

# ── Extended mode filters ──
start_date_input = widgets.DatePicker(
    description='Start date:',
    value=datetime(2025, 1, 1).date(),
    style=style
)

end_date_input = widgets.DatePicker(
    description='End date:',
    value=datetime.now().date(),
    style=style
)

date_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'The notebook will break this range into monthly windows and fetch up to 100 results per window. '
    'Each batch is saved as a checkpoint before combining into a single file.</p>'
)

# ── Common filters ──
country_input = widgets.Dropdown(
    options=[
        ('United States', 'US'), ('United Kingdom', 'GB'), ('Canada', 'CA'),
        ('Australia', 'AU'), ('India', 'IN'), ('Germany', 'DE'),
        ('France', 'FR'), ('Brazil', 'BR'), ('Mexico', 'MX'), ('Japan', 'JP'),
    ],
    value='US',
    description='Country:',
    layout=layout,
    style=style
)

lang_input = widgets.Dropdown(
    options=[
        ('English', 'en'), ('Spanish', 'es'), ('French', 'fr'),
        ('German', 'de'), ('Portuguese', 'pt'), ('Japanese', 'ja'),
    ],
    value='en',
    description='Language:',
    layout=layout,
    style=style
)

max_results_input = widgets.IntSlider(
    value=100, min=10, max=100, step=10,
    description='Max per batch:',
    style=style
)

# ── Export ──
export_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Export</h3>')

format_input = widgets.Dropdown(
    options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx)', 'xlsx')],
    value='csv',
    description='Format:',
    layout=layout,
    style=style
)

fetch_btn = widgets.Button(
    description='Fetch News',
    button_style='',
    layout=widgets.Layout(width='200px', margin='20px 0'),
    style={'button_color': '#6096BA'}
)

out = widgets.Output()

# ── Dynamic UI: show/hide fields based on mode ──
quick_box = widgets.VBox([period_input])
extended_box = widgets.VBox([widgets.HBox([start_date_input, end_date_input]), date_help])
extended_box.layout.display = 'none'

def on_mode_change(change):
    if change['new'] == 'quick':
        quick_box.layout.display = ''
        extended_box.layout.display = 'none'
    else:
        quick_box.layout.display = 'none'
        extended_box.layout.display = ''

mode_toggle.observe(on_mode_change, names='value')


def export_df(df, query):
    """Export DataFrame and return filepath."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    slug = query.replace(' ', '_')[:30] if query else 'top_news'
    base = f'google_news_{slug}_{timestamp}'
    fmt = format_input.value

    if fmt == 'xlsx':
        filepath = f'{base}.xlsx'
        df.to_excel(filepath, sheet_name='News Results', index=False, engine='openpyxl')
        style_excel(filepath)
    else:
        filepath = f'{base}.csv'
        df.to_csv(filepath, index=False)
    return filepath


def sort_by_date(df):
    """Sort DataFrame by parsed date, newest first."""
    try:
        df['date_parsed'] = pd.to_datetime(df['published'], format='mixed', utc=True)
        df = df.sort_values('date_parsed', ascending=False).drop(columns=['date_parsed'])
    except Exception:
        pass
    return df


def on_fetch_clicked(_):
    out.clear_output()
    with out:
        query = query_input.value.strip()
        mode = mode_toggle.value

        if mode == 'quick':
            # ── Quick Search ──
            is_top_news = not query
            gn = GNews(
                language=lang_input.value,
                country=country_input.value,
                period=period_input.value,
                max_results=max_results_input.value,
            )

            if is_top_news:
                print("\U0001f4f0 Fetching top headlines...")
            else:
                print(f"\U0001f50d Searching Google News for: {query}")
            print(f"   Country: {country_input.value}, Language: {lang_input.value}")
            print(f"   Period: {period_input.value}, Max results: {max_results_input.value}")
            print()

            try:
                results = gn.get_top_news() if is_top_news else gn.get_news(query)
                if not results:
                    print("\u26a0\ufe0f No results found. Try different search terms or a broader time period.")
                    return

                print(f"\u2713 Retrieved {len(results)} articles")
                print()

                df = pd.DataFrame(results_to_rows(results))
                df = sort_by_date(df)

                print("\U0001f4ca Preview:")
                display(df[['title', 'source', 'published']].head(15))

                filepath = export_df(df, query)
                print()
                print(f"\u2713 Saved: {filepath} ({len(df)} articles)")

                if IN_COLAB:
                    colab_files.download(filepath)
                    print(f"\U0001f4e5 Downloaded: {filepath}")

            except Exception as e:
                print(f"\u274c Error: {type(e).__name__}: {e}")

        else:
            # ── Extended Search ──
            if not query:
                print("\u26a0\ufe0f Extended Search requires keywords. Enter search terms above.")
                return

            start_dt = datetime.combine(start_date_input.value, datetime.min.time())
            end_dt = datetime.combine(end_date_input.value, datetime.min.time())

            if start_dt >= end_dt:
                print("\u26a0\ufe0f Start date must be before end date.")
                return

            # Build monthly windows
            windows = []
            current = start_dt
            while current < end_dt:
                window_end = current + relativedelta(months=1)
                if window_end > end_dt:
                    window_end = end_dt
                windows.append((current, window_end))
                current = window_end

            print(f"\U0001f50d Extended Search: {query}")
            print(f"   Date range: {start_dt.strftime('%b %d, %Y')} \u2013 {end_dt.strftime('%b %d, %Y')}")
            print(f"   Monthly windows: {len(windows)}")
            print(f"   Max per window: {max_results_input.value}")
            print(f"   Country: {country_input.value}, Language: {lang_input.value}")
            print()

            all_rows = []
            batch_files = []
            slug = query.replace(' ', '_')[:30]

            for i, (w_start, w_end) in enumerate(windows):
                label = w_start.strftime('%b %Y')
                print(f"   \U0001f4c5 {label}...", end=" ")

                try:
                    gn = GNews(
                        language=lang_input.value,
                        country=country_input.value,
                        max_results=max_results_input.value,
                        start_date=w_start,
                        end_date=w_end,
                    )
                    results = gn.get_news(query)
                    rows = results_to_rows(results)

                    # Save checkpoint
                    batch_path = f'batch_{slug}_{w_start.strftime("%Y%m")}.csv'
                    pd.DataFrame(rows).to_csv(batch_path, index=False)
                    batch_files.append(batch_path)

                    all_rows.extend(rows)
                    print(f"{len(results)} articles")

                except Exception as e:
                    print(f"error ({type(e).__name__}: {e})")

            print()

            if not all_rows:
                print("\u26a0\ufe0f No results found across any time window.")
                return

            # Combine and deduplicate
            df = pd.DataFrame(all_rows)
            before_dedup = len(df)
            df = df.drop_duplicates(subset=['url'])
            df = sort_by_date(df)

            print(f"\u2713 Combined: {before_dedup} total \u2192 {len(df)} unique articles")
            print(f"   Batch checkpoints saved: {len(batch_files)} files")
            print()

            print("\U0001f4ca Preview:")
            display(df[['title', 'source', 'published']].head(15))

            filepath = export_df(df, query)
            print()
            print(f"\u2713 Final export: {filepath} ({len(df)} articles)")

            if IN_COLAB:
                colab_files.download(filepath)
                print(f"\U0001f4e5 Downloaded: {filepath}")

            # Clean up batch files
            for bf in batch_files:
                try:
                    os.remove(bf)
                except Exception:
                    pass


fetch_btn.on_click(on_fetch_clicked)

display(widgets.VBox([
    instructions,
    mode_header,
    mode_toggle,
    search_header,
    query_input,
    query_help,
    config_header,
    quick_box,
    extended_box,
    country_input,
    lang_input,
    max_results_input,
    export_header,
    format_input,
    fetch_btn,
    out,
]))